# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook guides you step-by-step through loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library, referencing dataset entities by their `@id` fields throughout, as recommended for working with Croissant schemas.

### Dataset Source
The FAIR^2 dataset is described via a Croissant schema URL and contains record sets and fields on logistic regression results, survey demographics, and rangeland management in Northern Kenya.

Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install the mlcroissant library if needed
!pip install mlcroissant --quiet

## 1. Data Loading

Load the FAIR^2 dataset metadata and records into your environment using the `mlcroissant` Python API.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata (as an object, not subscripted)
md = dataset.metadata

# Display metadata summary
print(f"Dataset: {md.name}")
print(f"Description: {md.description}\n")
print(f"License: {md.license}")
print(f"Spatial Coverage: {md.spatialCoverage}")
print(f"Temporal Coverage: {md.temporalCoverage}")

## 2. Data Overview

List the record sets and fields defined in the dataset. This exploration will use the Croissant schema and the `@id` for each record set, field, and column as required for reproducibility and clarity.

In [ ]:
# List all record sets and their fields by @id

print("Available record sets in this dataset (with @id and name):\n")
record_sets = []
for rs in dataset.record_sets:
    print(f"@id: {rs.id} | name: {rs.name}")
    record_sets.append(rs.id)
    if hasattr(rs, 'fields'):
        for field in rs.fields:
            print(f"    field @id: {field.id} | name: {field.name}")

# For demonstration, show one record from each record set
print("\nExample record from each record set:")
for record_set_id in record_sets:
    try:
        record_iter = dataset.records(record_set=record_set_id)
        first_record = next(record_iter)
        print(f"\nRecord set @id: {record_set_id}")
        print(json.dumps(first_record, indent=2))
    except StopIteration:
        print(f"(No records found for {record_set_id})")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

## 3. Data Extraction

Extract entire contents for each record set into pandas DataFrames. Each DataFrame key references the record set's `@id`. To select fields or columns, always use their `@id` as well.

In [ ]:
# Prepare extraction of all record sets by @id
# (Record sets are identified by their full @id URIs)
dataframes = {}

for rs_id in record_sets:
    try:
        recs = list(dataset.records(record_set=rs_id))
        if recs:
            dataframes[rs_id] = pd.DataFrame(recs)
            print(f"Loaded DataFrame for record set @id: {rs_id} (shape: {dataframes[rs_id].shape})")
        else:
            print(f"No records for {rs_id}")
    except Exception as e:
        print(f"Error reading {rs_id}: {e}")

# Show columns of the first loaded DataFrame for inspection
if dataframes:
    main_rs = list(dataframes.keys())[0]
    print(f"\nColumns in the first available record set (@id={main_rs}):")
    print(dataframes[main_rs].columns.tolist())

    print(f"\nPreview of data in @id={main_rs}:")
    display(dataframes[main_rs].head())

## 4. Exploratory Data Analysis (EDA)

Carry out a basic analysis using numeric fields. Use only `@id` identifiers for fields and columns.

Below, we:
- Choose a numeric field by its `@id` (update as needed for your use case and based on prior overview)
- Filter for values above a threshold
- Normalize the field
- Optionally group by a categorical field's `@id`

In [ ]:
# EXAMPLE: You may need to adjust `numeric_field_id` and `group_field_id` to match available field @id's in your dataset.

# Get a numeric field's @id (edit as found in the Data Overview step)
main_rs = list(dataframes.keys())[0]  # Use the first available record set

# Try to select a numeric field (by @id) dynamically
df = dataframes[main_rs]

numeric_field_id = None
for col in df.columns:
    if "coeff" in col.lower() or "log_likelihood" in col.lower() or "std" in col.lower():
        numeric_field_id = col
        break

if numeric_field_id is None:
    # Fallback to first numeric column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col].dropna()):
            numeric_field_id = col
            break

if numeric_field_id is None:
    print("No obvious numeric field -- please set `numeric_field_id` manually from field @ids.")
else:
    print(f"Using numeric field @id: {numeric_field_id}")

    # Filter
    threshold = 0   # Adjust as appropriate for field
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records where {numeric_field_id} > {threshold} (count: {len(filtered_df)}):")
    display(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nAdded normalized field: {norm_col}")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Attempt to pick a group field by @id (categorical)
    group_field_id = None
    for col in df.columns:
        if ("ward" in col.lower() or "gender" in col.lower() or "region" in col.lower()) and not pd.api.types.is_numeric_dtype(df[col].dropna()):
            group_field_id = col
            break

    if group_field_id and group_field_id in filtered_df.columns:
        print(f"\nGrouping by field @id: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization

Plot data distributions or relationships between selected fields, referencing columns/fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and numeric_field_id and (len(filtered_df) > 0):
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field_id], kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group field, if available
    if 'group_field_id' in locals() and group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion

In this notebook, we loaded and explored the FAIR^2 ordered logistic regression dataset using `mlcroissant`, referencing all record sets and fields by their `@id`, and demonstrated example filtering, normalization, grouping, and visualizations. For further specialized analyses, continue with domain-specific variable choices using the schema-overview approach shown here.